In [8]:
import matplotlib.pyplot as plt
from IPython.display import SVG, display

from dataclasses import dataclass
from pathlib import Path
import csv
import cmocean
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import numpy as np
from desmin_reconstruction.plotting import save_panel
from desmin_reconstruction.utils.io import load_localizations
from IPython.display import display, SVG


PROTEINS = ("desmin", "actinin")
COLOR_DICT = {"desmin": "#CC3311", "actinin": "#009988"}
LABEL_DICT = {"desmin": "Desmin", "actinin": "α-actinin"}

In [4]:
# Read the original coordinates from CSV.
points = load_localizations(
    "../../source_data/fig_2/", prefix="2g_"
)

In [ ]:
"""Two protein rows, four cumulative z ranges: PyVista -> Matplotlib.

Dependencies: numpy, matplotlib, pyvista; colorcet for the original palette.
CSV columns must be x (um), y (um), z (um). Coordinates remain in micrometers.

Render once with render_depth_series(); restyle with compose_depth_series().
The microscopy images are raster; text, ticks and axes export as vectors.
"""


def select_z_range(xyz, upper, *, z_reference=0.3, exclude_below_reference=False):
    """Match the original strict z < upper selection.

    By default points below z_reference are retained, as in the supplied code.
    Set exclude_below_reference=True to select z_reference <= z < upper.
    """
    keep = xyz[:, 2] < upper
    if exclude_below_reference:
        keep &= xyz[:, 2] >= z_reference
    return xyz[keep]


@dataclass
class RenderedDepthSeries:
    images: tuple                 # images[row][column], RGB, top row first
    extent: tuple                 # requested xmin, xmax, ymin, ymax
    image_extent: tuple           # camera extent after pixel-width rounding
    z_edges: tuple
    z_reference: float
    clim: tuple
    cmaps: dict
    counts: np.ndarray            # selected localizations, before XY clipping


def _resolve_cmaps(cmaps):
    if cmaps is None:
        import colorcet as cc
        cmaps = {"desmin": "autumn", "actinin": cc.cm.bgy_r}
    resolved = {}
    for protein in PROTEINS:
        cmap = cmaps[protein]
        resolved[protein] = mpl.colormaps[cmap] if isinstance(cmap, str) else cmap
        if not isinstance(resolved[protein], mpl.colors.Colormap):
            raise TypeError(f"{protein}: supply a Matplotlib colormap or its name.")
    return resolved


def render_depth_series(
    points,
    *,
    cmaps=None,
    z_edges=(0.3, 0.9, 1.5, 2.1, 2.8),
    z_reference=0.3,
    clim=(0.0, 2.5),
    xy_bounds=(0.5, 10.5, 2.0, 12.0),
    height_px=1400,
    point_size=3.0,
    opacity=0.4,
    exclude_below_reference=False,
):
    """Render cumulative z views with identical XY cameras and color limits.

    The default reproduces the original autumn / Colorcet bgy_r colormaps.
    The four cutoffs are z_edges[1:]. Titles use cutoff - z_edges[0].
    Color is always the original z coordinate minus z_reference.

    height_px controls screenshot resolution. point_size is passed unchanged
    to PyVista's Gaussian mapper; its sizing is not ordinary pixel-sized
    scatter sizing. XYZ coordinates are not projected, normalized or re-zeroed.
    """
    import pyvista as pv

    points = {name: np.asarray(points[name], dtype=float) for name in PROTEINS}
    for name, xyz in points.items():
        if xyz.ndim != 2 or xyz.shape[1] != 3 or len(xyz) == 0:
            raise ValueError(f"{name}: expected a nonempty (N, 3) array.")
        if not np.isfinite(xyz).all():
            raise ValueError(f"{name}: all coordinates must be finite.")
    edges = np.asarray(z_edges, dtype=float)
    if edges.ndim != 1 or len(edges) < 2 or not np.isfinite(edges).all() or np.any(np.diff(edges) <= 0):
        raise ValueError("z_edges must be finite and strictly increasing.")
    extent = np.asarray(xy_bounds, dtype=float)
    if extent.shape != (4,) or not np.isfinite(extent).all():
        raise ValueError("xy_bounds must contain four finite coordinates.")
    x0, x1, y0, y1 = extent
    dx, dy = x1 - x0, y1 - y0
    if dx <= 0 or dy <= 0:
        raise ValueError("XY bounds must be increasing.")
    clim = tuple(float(v) for v in clim)
    if len(clim) != 2 or not np.isfinite(clim).all() or clim[0] >= clim[1]:
        raise ValueError("clim must contain increasing finite limits.")
    if not np.isfinite(z_reference) or point_size <= 0 or not 0 <= opacity <= 1:
        raise ValueError("Check z_reference, point_size and opacity.")
    height_px = int(height_px)
    if height_px < 2:
        raise ValueError("height_px must be at least 2.")
    width_px = max(2, int(np.ceil(height_px * dx / dy)))
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    zlo = min(xyz[:, 2].min() for xyz in points.values())
    zhi = max(xyz[:, 2].max() for xyz in points.values())
    cz = (zlo + zhi) / 2
    distance = 2 * max(dx, dy, zhi - zlo)
    camera_position = [(cx, cy, cz + distance), (cx, cy, cz), (0, 1, 0)]
    half_width = (dy / 2) * width_px / height_px
    image_extent = (cx - half_width, cx + half_width, y0, y1)
    cmaps = _resolve_cmaps(cmaps)
    images = []
    counts = np.zeros((2, len(edges) - 1), dtype=int)

    p = pv.Plotter(off_screen=True, window_size=(width_px, height_px))
    try:
        for row, protein in enumerate(PROTEINS):
            row_images = []
            for col, upper in enumerate(edges[1:]):
                xyz = select_z_range(
                    points[protein], upper, z_reference=z_reference,
                    exclude_below_reference=exclude_below_reference,
                )
                counts[row, col] = len(xyz)
                # Avoid attempting to add an empty PolyData.
                if len(xyz) == 0:
                    row_images.append(np.zeros((height_px, width_px, 3), dtype=np.uint8))
                    continue
                p.clear()
                p.set_background("black")
                cloud = pv.PolyData(xyz)
                cloud.point_data["z_relative"] = xyz[:, 2] - z_reference
                p.add_mesh(
                    cloud, scalars="z_relative", cmap=cmaps[protein],
                    clim=clim, n_colors=256, style="points_gaussian",
                    point_size=point_size, opacity=opacity,
                    show_scalar_bar=False, reset_camera=False,
                )
                p.enable_parallel_projection()
                p.camera_position = camera_position
                p.camera.parallel_scale = dy / 2
                p.reset_camera_clipping_range()
                row_images.append(np.array(
                    p.screenshot(transparent_background=False), copy=True,
                ))
            images.append(tuple(row_images))
    finally:
        p.close()
    return RenderedDepthSeries(
        tuple(images), tuple(extent), image_extent, tuple(edges),
        float(z_reference), clim, cmaps, counts,
    )


def compose_depth_series(
    rendered,
    *,
    figsize=(11.6, 5.9),
    margins=(1.05, 0.72, 0.65, 0.46),  # left, right, bottom, top, INCHES
    panel_gap=0.12, row_gap=0.18,
    colorbar_gap=0.14, colorbar_width=0.13,
    xticks=(2, 4, 6, 8, 10), yticks=(2, 4.5, 7, 9.5, 12),
    x_tick_format="%g", y_tick_format="%g",
    axis_label_size=16, tick_label_size=11,
    title_size=13, protein_label_size=14,
    colorbar_label_size=14, colorbar_tick_size=11,
    tick_pad=4, title_pad=8,
    xlabel_offset=0.42, ylabel_offset=0.91, protein_label_offset=0.63,
    row_labels=None, identity_colors=None,
    spine_color="#888888", font_family="DejaVu Sans",
):
    """Compose a full canvas; all layout distances except pads are in inches.

    figsize is independently adjustable. Equal spatial aspect is retained:
    extra width/height becomes black margin rather than stretching the image.
    Font sizes and tick_pad/title_pad are in points. X ticks are centered.
    Other columns retain their Y ticks but hide numeric Y labels.
    Returns (fig, axes, colorbars); axes has shape (2, number_of_cutoffs).
    """
    fw, fh = map(float, figsize)
    left, right, bottom, top = map(float, margins)
    ncols = len(rendered.z_edges) - 1
    x0, x1, y0, y1 = rendered.extent
    aspect = (x1 - x0) / (y1 - y0)
    usable_w = fw - left - right - (ncols - 1) * panel_gap - colorbar_gap - colorbar_width
    usable_h = fh - bottom - top - row_gap
    if min(fw, fh, usable_w, usable_h, colorbar_width) <= 0:
        raise ValueError("Figure too small for the supplied margins and gaps.")
    if min(left, right, bottom, top, panel_gap, row_gap, colorbar_gap) < 0:
        raise ValueError("Margins and gaps must be nonnegative.")
    panel_h = min(usable_h / 2, usable_w / (ncols * aspect))
    panel_w = panel_h * aspect
    start_x = left + (usable_w - ncols * panel_w) / 2
    start_y = bottom + (usable_h - 2 * panel_h) / 2
    image_row_w = ncols * panel_w + (ncols - 1) * panel_gap
    row_labels = LABEL_DICT if row_labels is None else row_labels
    identity_colors = COLOR_DICT if identity_colors is None else identity_colors
    norm = mpl.colors.Normalize(*rendered.clim)
    colorbars = []
    axes = np.empty((2, ncols), dtype=object)

    with plt.style.context("default"), mpl.rc_context({
        "font.family": font_family, "text.color": "white",
        "axes.labelcolor": "white", "xtick.color": "white",
        "ytick.color": "white", "figure.autolayout": False,
        "figure.constrained_layout.use": False,
    }):
        fig = plt.figure(figsize=(fw, fh), facecolor="black")
        for row, protein in enumerate(PROTEINS):
            y = start_y + (1 - row) * (panel_h + row_gap)
            for col, upper in enumerate(rendered.z_edges[1:]):
                x = start_x + col * (panel_w + panel_gap)
                shared = {} if row == col == 0 else dict(sharex=axes[0, 0], sharey=axes[0, 0])
                ax = fig.add_axes(
                    [x / fw, y / fh, panel_w / fw, panel_h / fh],
                    facecolor="black", **shared,
                )
                axes[row, col] = ax
                # A screenshot's first row is its top, hence origin='upper'.
                ax.imshow(
                    rendered.images[row][col], extent=rendered.image_extent,
                    origin="upper", interpolation="none", aspect="equal",
                )
                ax.set_xticks(xticks)
                ax.set_yticks(yticks)
                ax.set_xlim(x0, x1)
                ax.set_ylim(y0, y1)
                ax.xaxis.set_major_formatter(FormatStrFormatter(x_tick_format))
                ax.yaxis.set_major_formatter(FormatStrFormatter(y_tick_format))
                ax.tick_params(
                    axis="both", direction="in", length=3, width=0.7,
                    color=spine_color, labelcolor="white", labelsize=tick_label_size,
                    pad=tick_pad, labelbottom=(row == 1), labelleft=(col == 0),
                    top=False, right=False,
                )
                for side in ("top", "right"):
                    ax.spines[side].set_visible(False)
                for side in ("bottom", "left"):
                    ax.spines[side].set_color(spine_color)
                    ax.spines[side].set_linewidth(0.7)
                # Lift a boundary Y label away from the bottom X labels.
                if col == 0:
                    for value, label in zip(ax.get_yticks(), ax.get_yticklabels()):
                        if np.isclose(value, y0):
                            label.set_va("bottom")
                        elif np.isclose(value, y1):
                            label.set_va("top")
                if row == 0:
                    thickness = upper - rendered.z_edges[0]
                    ax.set_title(f"Thickness: {thickness:.1f} μm", fontsize=title_size, pad=title_pad)
            fig.text(
                (start_x - protein_label_offset) / fw, (y + panel_h / 2) / fh,
                row_labels[protein], rotation=90, ha="center", va="center",
                color=identity_colors[protein], fontsize=protein_label_size,
            )
            cax = fig.add_axes([
                (start_x + image_row_w + colorbar_gap) / fw,
                y / fh, colorbar_width / fw, panel_h / fh,
            ])
            sm = mpl.cm.ScalarMappable(norm=norm, cmap=rendered.cmaps[protein])
            cb = fig.colorbar(sm, cax=cax, ticks=np.linspace(*rendered.clim, 6))
            cb.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
            cb.ax.tick_params(colors="white", labelsize=colorbar_tick_size, length=0, pad=4)
            cb.outline.set_edgecolor(spine_color)
            cb.outline.set_linewidth(0.6)
            cb.set_label("z (μm)", color="white", fontsize=colorbar_label_size, labelpad=5)
            # Keep the colorbar vector in SVG/PDF too.
            if cb.solids is not None:
                cb.solids.set_rasterized(False)
            colorbars.append(cb)
        fig.supxlabel(
            "x (μm)", x=(start_x + image_row_w / 2) / fw,
            y=(start_y - xlabel_offset) / fh,
            fontsize=axis_label_size, va="center",
        )
        fig.supylabel(
            "y (μm)", x=(start_x - ylabel_offset) / fw,
            y=(start_y + panel_h + row_gap / 2) / fh,
            fontsize=axis_label_size, ha="center",
        )
    return fig, axes, tuple(colorbars)


In [19]:
CMAP_DICT = {
    "desmin": cmocean.tools.crop_by_percent(
        cmocean.cm.amp_r, 15, which="min"
    ),
    "actinin": cmocean.tools.crop_by_percent(
        cmocean.cm.tempo_r, 15, which="min"
    ),
}

# 2. Render once. Defaults retain autumn and Colorcet bgy_r.
rendered = render_depth_series(
    points,
    cmaps = CMAP_DICT,
    z_edges=(0.3, 0.9, 1.5, 2.1, 2.8),
    z_reference=0.3,                 # Color represents z - 0.3
    clim=(0.0, 2.5),                  # Fixed across all eight panels
    xy_bounds=(0.5, 10.5, 2.0, 12.0),
    height_px=1400,                   # Resolution of each microscopy image
    point_size=3.0,
    opacity=0.4,
)

# 3. Assemble. Rerun this section to adjust the presentation.
fig, axes, colorbars = compose_depth_series(
    rendered,
    figsize=(11.6, 5.9),             # Width, height in inches
    margins=(1.05, 0.72, 0.65, 0.46), # Left, right, bottom, top: inches
    panel_gap=0.12,                   # Horizontal gap: inches
    row_gap=0.18,                     # Vertical gap: inches

    xticks=(2, 4, 6, 8, 10),
    yticks=(2, 4.5, 7, 9.5, 12),

    axis_label_size=16,
    tick_label_size=13,
    title_size=14,
    protein_label_size=16,
    colorbar_label_size=14,
    colorbar_tick_size=13,

    xlabel_offset=0.42,               # Distance below panels: inches
    ylabel_offset=0.91,               # Distance left of panels: inches
    protein_label_offset=0.63,
)

In [21]:
savepath = "../../figures/2g"
save_panel(fig, savepath, formats=("svg",))
display(SVG(filename=str(savepath + ".svg")))
plt.close(fig)